# 🚀 Step 2 — Multi-Model LLM Server on Kaggle GPU

Runs an **OpenAI-compatible LLM server** with two selectable models on Kaggle's free GPU, exposed via a Cloudflare tunnel:

| Model alias | Model | Use |
|---|---|---|
| `qwen3.6-35b-a3b` | Qwen3.6-35B-A3B (MoE, multimodal) | General chat, vision, reasoning |
| `deepseek-v4-pro-9b` | DeepSeek-V4-Pro-Qwen3.5-9B (uncensored) | Math/STEM, fast, uncensored |

### How model switching works
```
Your Agent (Codex / Cline / Continue / Open WebUI / chat.py ...)
   │  OpenAI API, one URL, choose model by "model" field
   ▼
Cloudflare Tunnel  →  Gateway :8080  →  llama-server :8081 (loads the chosen model on demand)
```
Clients **always use the same URL**; switching = change the `model` name. Only one model is in VRAM at a time (full speed), switching loads the other (~10s–2min).

In the **built-in WebUI**, switching is one click: open the model selector in the chat composer, or use "Start a new chat" from the Models page — both call `/models/load` and the gateway loads that model. No extra scripts needed; everything is wired up here.

> 🔧 Auto-detects username and dataset paths. Attach the datasets and click **Run All** — one pass sets up the gateway, the official WebUI, the tunnel, and model switching.

### Prerequisites
- Accelerator **GPU T4 x2**, Internet **ON**
- Datasets attached: `qwen36-35b-a3b-gguf`, `llama-cpp-bin`, `deepseek-v4-pro-9b-gguf`

## ⚙️ Cell 1 — Configuration

All settings here. Both models are auto-discovered from attached datasets. Missing one is OK — the server will expose only the models it finds.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit only if you want to change defaults
# ══════════════════════════════════════════════════════════════════════════════
KAGGLE_USERNAME = ""   # leave empty = auto-detect; or fill in your own username

PORT       = 8080          # gateway port (clients connect here)
BACKEND_PORT = 8081        # llama-server backend (internal)
CTX_SIZE   = 16384         # for the 35B model
CTX_SIZE_9B = 32768        # for the 9B model (smaller model can afford more)
N_GPU_LAYERS = 999

# Sampling defaults (non-thinking). Agents may override per request.
TEMPERATURE, TOP_P, TOP_K, MIN_P, PRESENCE_PENALTY = 0.7, 0.8, 20, 0.0, 1.5

# Model aliases — these are the exact strings agents put in the "model" field.
ALIAS_QWEN = "qwen3.6-35b-a3b"
ALIAS_DEEPSEEK = "deepseek-v4-pro-9b"

# ══════════════════════════════════════════════════════════════════════════════
import os, json
from pathlib import Path

WORKING      = Path("/kaggle/working")
LLAMA_SERVER = WORKING / "llama-server"
INPUT_ROOT   = Path("/kaggle/input")

def _detect_kaggle_username():
    """Best-effort detection of the current Kaggle username. Returns "" if unknown."""
    # 1) Environment variables set by Kaggle
    for key in ("KAGGLE_USERNAME", "KAGGLE_USER", "KAGGLE_KERNEL_OWNER_NAME", "KERNEL_OWNER"):
        u = os.environ.get(key, "").strip()
        if u and u.lower() not in ("unknown", "none"):
            return u
    # 2) Kaggle API token (~/.kaggle/kaggle.json)
    for token in (Path.home()/".kaggle"/"kaggle.json",
                  Path("/root/.kaggle/kaggle.json"),
                  Path("/kaggle/working/.kaggle/kaggle.json")):
        try:
            if token.exists():
                d = json.loads(token.read_text())
                u = str(d.get("username", "")).strip()
                if u:
                    return u
        except Exception:
            pass
    # 3) Attached dataset path: /kaggle/input/datasets/<owner>/<slug>
    for p in Path("/kaggle/input").glob("datasets/*/*"):
        parts = p.parts
        if "datasets" in parts:
            i = parts.index("datasets")
            if i + 1 < len(parts):
                return parts[i+1]
    return ""

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = _detect_kaggle_username()

def find_file(*glob_patterns):
    for pat in glob_patterns:
        for p in INPUT_ROOT.glob(pat):
            if p.is_file():
                return p
    return None

# ── Locate Qwen model + mmproj ────────────────────────────────────────────────
qwen_model  = find_file("**/Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf", "**/*Qwen3.6*Q4_K_XL*.gguf")
qwen_mmproj = find_file("**/mmproj-F16.gguf")

# ── Locate DeepSeek 9B model + mmproj ─────────────────────────────────────────
# Search main model by exact quant names first, explicitly EXCLUDING any mmproj
# (the projector filename also contains DeepSeek/9B/heretic and would otherwise
# be picked up by a loose wildcard). Works with F16 / f16_q8_0 / Q4_K_M.
def find_ds_model():
    for pat in (
        "**/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-f16.gguf",
        "**/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-f16_q8_0.gguf",
        "**/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-Q4_K_M.gguf",
        "**/*DeepSeek*9B*heretic*.gguf",
    ):
        for p in INPUT_ROOT.glob(pat):
            if p.is_file() and "mmproj" not in p.name.lower():
                return p
    return None

ds_model  = find_ds_model()
ds_mmproj = find_file("**/mmproj-DeepSeek*heretic*.gguf")

# ── Build the model registry the gateway will use ─────────────────────────────
MODEL_CONFIGS = {}
if qwen_model:
    MODEL_CONFIGS[ALIAS_QWEN] = {
        "model": str(qwen_model),
        "mmproj": str(qwen_mmproj) if qwen_mmproj else None,
        "ctx": CTX_SIZE, "ngl": N_GPU_LAYERS,
        "temp": TEMPERATURE, "top_p": TOP_P, "top_k": TOP_K,
        "min_p": MIN_P, "presence": PRESENCE_PENALTY,
    }
if ds_model:
    MODEL_CONFIGS[ALIAS_DEEPSEEK] = {
        "model": str(ds_model),
        "mmproj": str(ds_mmproj) if ds_mmproj else None,
        "ctx": CTX_SIZE_9B, "ngl": N_GPU_LAYERS,
        "temp": TEMPERATURE, "top_p": TOP_P, "top_k": TOP_K,
        "min_p": MIN_P, "presence": PRESENCE_PENALTY,
    }

if not MODEL_CONFIGS:
    raise FileNotFoundError(
        "No model files found. Attach at least one dataset: qwen36-35b-a3b-gguf "
        "or deepseek-v4-pro-9b-gguf, then restart the kernel and Run All."
    )

DEFAULT_MODEL = ALIAS_QWEN if ALIAS_QWEN in MODEL_CONFIGS else next(iter(MODEL_CONFIGS))

print("✅ Configuration ready")
print(f"   Kaggle user : {KAGGLE_USERNAME or '(auto)'}")
print(f"   Models      : {list(MODEL_CONFIGS.keys())}")
print(f"   Default     : {DEFAULT_MODEL}")
for a, cfg in MODEL_CONFIGS.items():
    print(f"     - {a}: {cfg['model']}")
    if cfg.get("mmproj"):
        print(f"         mmproj: {cfg['mmproj']}")

## 🖥️ Cell 2 — GPU check

In [ ]:
import subprocess

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("❌ No GPU detected. Settings → Accelerator → GPU T4 x2.")
else:
    total_vram = 0
    print(f"{'GPU':<5} {'Name':<20} {'Total VRAM':>12} {'Free VRAM':>12}")
    print("-" * 55)
    for i, line in enumerate(result.stdout.strip().split("\n")):
        name, total, free = [x.strip() for x in line.split(",")]
        tm = int(total.replace(" MiB", "")); fm = int(free.replace(" MiB", ""))
        total_vram += tm
        print(f"  {i:<3} {name:<20} {tm/1024:>10.1f} GB {fm/1024:>10.1f} GB")
    print("-" * 55)
    print(f"  Total VRAM: {total_vram/1024:.1f} GB")
    for alias, cfg in MODEL_CONFIGS.items():
        size = Path(cfg["model"]).stat().st_size / (1024**3)
        print(f"  {alias}: {size:.1f} GB model" + (" (+mmproj)" if cfg.get("mmproj") else ""))
    print("  Note: models load one at a time, so each gets full VRAM.")
    print("  F16 9B (~17 GB) loads alone in 30 GB VRAM; switch = unload other model.")

## 📦 Cell 3 — Load llama.cpp binaries

As before: copy the compiled binaries + `.so` libraries from the `llama-cpp-bin` dataset.

> ⚠️ The `qwen3_5` model needs llama.cpp **b12182+**. If loading fails later, re-run `1b_setup_llama.ipynb` to rebuild with the latest master.

In [ ]:
import shutil, os, subprocess

binaries = ["llama-server", "llama-cli", "llama-mtmd-cli"]
for name in binaries:
    src_file = None
    for p in INPUT_ROOT.glob("**/" + name):
        if p.is_file():
            src_file = p; break
    if src_file is None:
        raise FileNotFoundError(f"Binary '{name}' not found. Re-run 1b_setup_llama.ipynb.")
    dst = WORKING / name
    shutil.copy2(src_file, dst)
    dst.chmod(0o755)

so_count = 0
for p in INPUT_ROOT.glob("**/*.so*"):
    if p.is_file():
        shutil.copy2(p, WORKING / p.name)
        so_count += 1

os.environ["LD_LIBRARY_PATH"] = "/kaggle/working:" + os.environ.get("LD_LIBRARY_PATH", "")

test = subprocess.run([str(LLAMA_SERVER), "--version"], capture_output=True, text=True,
                      env={**os.environ, "LD_LIBRARY_PATH": os.environ["LD_LIBRARY_PATH"]})
print(test.stdout.strip() or test.stderr.strip())
print(f"\n✅ Binaries loaded ({len(binaries)} binaries + {so_count} libraries)")

## 🌐 Cell 4 — Install Cloudflare Tunnel

In [ ]:
%%bash
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
     -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
echo "cloudflared installed: $(cloudflared --version)"

## 🧠 Cell 5 — Write the multi-model gateway & start it

The gateway listens on port **8080** (this is the URL clients use). It reads each request's `model` field, loads that model into the backend, and proxies the request (including streaming). Changing models requires no client reconfiguration.

In [ ]:
# Write the gateway script (self-contained, no external files needed)
GATEWAY_SRC = r'''#!/usr/bin/env python3
# OpenAI-compatible multi-model switching gateway for Kaggle.
# Serves llama.cpp's OFFICIAL built-in WebUI by proxying it from the backend,
# while routing /v1/* API calls through the model-switching layer.
# Falls back to a built-in chat page if the backend has no WebUI.
import json, os, subprocess, threading, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import requests

GATEWAY_PORT = int(os.environ.get("GATEWAY_PORT", "8080"))
BACKEND_PORT = int(os.environ.get("BACKEND_PORT", "8081"))
LLAMA_SERVER = os.environ.get("LLAMA_SERVER", "/kaggle/working/llama-server")
MODEL_CONFIGS = json.loads(os.environ.get("MODEL_CONFIGS", "{}"))
DEFAULT_MODEL = os.environ.get("DEFAULT_MODEL", next(iter(MODEL_CONFIGS), ""))
LOG_PATH = "/tmp/llama-backend.log"

FALLBACK_UI = '<!DOCTYPE html>\n<html lang="zh">\n<head>\n<meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Kaggle LLM</title>\n<style>\n:root{--bg:#ffffff;--panel:#fbfbfb;--border:#e8e8e8;--text:#1f1f1f;--muted:#9a9a9a;--bubble:#f4f4f4;--accent:#1a1a1a}\n*{box-sizing:border-box}\nhtml,body{height:100%}\nbody{margin:0;font-family:ui-sans-serif,system-ui,-apple-system,"Segoe UI",Roboto,"Helvetica Neue",Arial,sans-serif;color:var(--text);background:var(--bg);display:flex;font-size:14px}\n#side{width:58px;flex:0 0 58px;background:var(--panel);border-right:1px solid var(--border);display:flex;flex-direction:column;align-items:center;padding:14px 0;gap:8px}\n#side .logo{width:34px;height:34px;border-radius:10px;background:#1a1a1a;color:#fff;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:16px;margin-bottom:8px}\n#side .ico{width:38px;height:38px;border-radius:10px;display:flex;align-items:center;justify-content:center;cursor:pointer;color:#666;font-size:19px;user-select:none}\n#side .ico:hover{background:#efefef}\n#side .spacer{flex:1}\n#main{flex:1;display:flex;flex-direction:column;height:100vh;position:relative;min-width:0}\n#wrap{flex:1;overflow-y:auto;display:flex;flex-direction:column;align-items:center}\n#welcome{margin:auto;text-align:center;padding:24px}\n#welcome h1{font-size:30px;font-weight:600;margin:0 0 10px;letter-spacing:-.4px}\n#welcome p{color:var(--muted);margin:0;font-size:15px}\n#msgs{width:min(760px,92%);display:none;flex-direction:column;gap:24px;padding:30px 0 168px}\n.msg{line-height:1.72;white-space:pre-wrap;word-break:break-word}\n.msg .who{font-size:12px;color:var(--muted);margin-bottom:6px}\n.msg.user{align-self:flex-end;max-width:82%;background:var(--bubble);padding:10px 16px;border-radius:18px}\n.msg.user .who{display:none}\n.msg.ai{align-self:flex-start;max-width:100%}\n#composer{position:absolute;left:0;right:0;bottom:0;padding:0 20px 20px;display:flex;justify-content:center;pointer-events:none}\n#box{pointer-events:auto;width:min(760px,92%);background:#fff;border:1px solid #dcdcdc;border-radius:26px;box-shadow:0 6px 26px rgba(0,0,0,.07);padding:14px 14px 9px}\n#box textarea{width:100%;border:0;outline:0;resize:none;font:inherit;font-size:15px;line-height:1.5;max-height:160px;min-height:28px;padding:2px 6px;background:transparent;color:var(--text)}\n#row{display:flex;align-items:center;gap:8px;margin-top:6px}\n.iconbtn{width:32px;height:32px;border-radius:50%;border:1px solid var(--border);background:#fff;cursor:pointer;display:flex;align-items:center;justify-content:center;font-size:18px;color:#555}\n#modelBtn{margin-left:auto;display:flex;align-items:center;gap:7px;border:1px solid var(--border);background:#fafafa;border-radius:18px;padding:6px 12px;cursor:pointer;font-size:13px;color:#333}\n#modelBtn .dot{width:7px;height:7px;border-radius:50%;background:#22a06b}\n#send{width:34px;height:34px;border-radius:50%;border:0;background:var(--accent);color:#fff;cursor:pointer;display:flex;align-items:center;justify-content:center;font-size:15px}\n#send:disabled{opacity:.45;cursor:not-allowed}\n#menu{position:absolute;bottom:70px;left:50%;transform:translateX(-50%);background:#fff;border:1px solid var(--border);border-radius:14px;box-shadow:0 8px 30px rgba(0,0,0,.14);padding:6px;min-width:250px;display:none;z-index:5}\n#menu div{padding:10px 12px;border-radius:9px;cursor:pointer;font-size:14px}\n#menu div:hover{background:#f2f2f2}\n#menu div.on{background:#1a1a1a;color:#fff}\n#hint{font-size:11px;color:#b5b5b5;text-align:center;margin-top:7px}\n</style>\n</head>\n<body>\n<div id="side">\n  <div class="logo" title="Kaggle LLM">K</div>\n  <div class="ico" id="newChat" title="新对话">＋</div>\n  <div class="ico" id="actModel" title="模型">◈</div>\n  <div class="spacer"></div>\n  <div class="ico" title="设置">⚙</div>\n</div>\n<div id="main">\n  <div id="wrap">\n    <div id="welcome">\n      <h1>Hello there</h1>\n      <p>Type a message or upload files to get started</p>\n    </div>\n    <div id="msgs"></div>\n  </div>\n  <div id="menu"></div>\n  <div id="composer">\n    <div id="box">\n      <textarea id="in" rows="1" placeholder="Type a message..."></textarea>\n      <div id="row">\n        <button class="iconbtn" id="plus" title="上传文件（暂未开放）">＋</button>\n        <button id="modelBtn"><span class="dot"></span><span id="modelLabel">加载中…</span><span style="color:#999">▾</span></button>\n        <button id="send" title="发送">➤</button>\n      </div>\n      <div id="hint">点输入框右下角模型标签切换模型，不用改网址</div>\n    </div>\n  </div>\n</div>\n<script>\nconst $=s=>document.querySelector(s);\nlet hist=[], current=\'\', busy=false;\nfunction scroll(){const w=$(\'#wrap\');w.scrollTop=w.scrollHeight}\nfunction setModel(m){\n  current=m;\n  $(\'#modelLabel\').textContent=m.replace(/^.*?:/,\'\');\n  document.querySelectorAll(\'#menu div\').forEach(d=>d.classList.toggle(\'on\',d.dataset.id===m));\n}\nfunction reset(){hist=[];$(\'#msgs\').innerHTML=\'\';$(\'#msgs\').style.display=\'none\';$(\'#welcome\').style.display=\'block\';$(\'#menu\').style.display=\'none\'}\nasync function loadModels(){\n  try{\n    const j=await (await fetch(\'/v1/models\')).json();\n    const menu=$(\'#menu\');menu.innerHTML=\'\';\n    (j.data||[]).forEach(m=>{\n      const d=document.createElement(\'div\');d.dataset.id=m.id;d.textContent=m.id;\n      d.onclick=()=>{setModel(m.id);reset()};\n      menu.appendChild(d);\n    });\n    setModel((j.data[0]||{}).id||\'\');\n  }catch(e){$(\'#modelLabel\').textContent=\'未就绪\'}\n}\nfunction addMsg(who,text){\n  $(\'#welcome\').style.display=\'none\';\n  const box=$(\'#msgs\');box.style.display=\'flex\';\n  const d=document.createElement(\'div\');d.className=\'msg \'+who;\n  const w=document.createElement(\'div\');w.className=\'who\';w.textContent=who===\'user\'?\'You\':current;\n  const t=document.createElement(\'div\');t.textContent=text;\n  d.appendChild(w);d.appendChild(t);box.appendChild(d);scroll();\n  return t;\n}\nasync function send(){\n  if(busy)return;\n  const v=$(\'#in\').value.trim();if(!v)return;\n  $(\'#in\').value=\'\';$(\'#in\').style.height=\'auto\';\n  addMsg(\'user\',v);hist.push({role:\'user\',content:v});\n  const out=addMsg(\'ai\',\'思考中…（首次加载模型可能要 1~2 分钟）\');\n  busy=true;$(\'#send\').disabled=true;\n  try{\n    const r=await fetch(\'/v1/chat/completions\',{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify({model:current,messages:hist})});\n    const j=await r.json();\n    if(j.error)throw new Error(j.error.message);\n    const a=j.choices[0].message.content;\n    out.textContent=a;hist.push({role:\'assistant\',content:a});\n  }catch(e){out.textContent=\'❌ \'+e.message}\n  busy=false;$(\'#send\').disabled=false;scroll();\n}\n$(\'#send\').onclick=send;\n$(\'#in\').onkeydown=e=>{if(e.key===\'Enter\'&&!e.shiftKey){e.preventDefault();send()}};\n$(\'#in\').oninput=()=>{$(\'#in\').style.height=\'auto\';$(\'#in\').style.height=Math.min($(\'#in\').scrollHeight,160)+\'px\'};\nfunction toggleMenu(){const m=$(\'#menu\');m.style.display=m.style.display===\'block\'?\'none\':\'block\'}\n$(\'#modelBtn\').onclick=e=>{e.stopPropagation();toggleMenu()};\n$(\'#actModel\').onclick=e=>{e.stopPropagation();toggleMenu()};\ndocument.addEventListener(\'click\',e=>{if(!e.target.closest(\'#menu\')&&!e.target.closest(\'#modelBtn\')&&!e.target.closest(\'#actModel\')) $(\'#menu\').style.display=\'none\'});\n$(\'#newChat\').onclick=reset;\n$(\'#plus\').onclick=()=>alert(\'文件上传暂未开放，请先输入文字消息。\');\nloadModels();\n</script>\n</body>\n</html>'

_lock = threading.Lock()
_current = None
_proc = None

def log(msg):
    print("[gateway] " + str(msg), flush=True)

def healthy():
    try:
        r = requests.get("http://localhost:%d/health" % BACKEND_PORT, timeout=3)
        return r.status_code == 200 and r.json().get("status") == "ok"
    except Exception:
        return False

def stop_backend():
    global _proc
    subprocess.run(["pkill", "-f", "llama-server"], capture_output=True)
    if _proc is not None and _proc.poll() is None:
        try:
            _proc.terminate()
            _proc.wait(timeout=20)
        except Exception:
            _proc.kill()
    _proc = None
    time.sleep(2)

def start_backend(alias):
    global _proc, _current
    cfg = MODEL_CONFIGS[alias]
    cmd = [
        LLAMA_SERVER,
        "--model", cfg["model"],
        "--host", "0.0.0.0",
        "--port", str(BACKEND_PORT),
        "--alias", alias,
        "--n-gpu-layers", str(cfg.get("ngl", 999)),
        "--ctx-size", str(cfg.get("ctx", 16384)),
        "--temp", str(cfg.get("temp", 0.7)),
        "--top-p", str(cfg.get("top_p", 0.8)),
        "--top-k", str(cfg.get("top_k", 20)),
        "--min-p", str(cfg.get("min_p", 0.0)),
        "--presence-penalty", str(cfg.get("presence", 1.5)),
        "--log-file", LOG_PATH,
    ]
    if cfg.get("mmproj"):
        cmd += ["--mmproj", cfg["mmproj"]]
    for extra in cfg.get("extra_args", []):
        cmd += [extra]
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = "/kaggle/working:" + env.get("LD_LIBRARY_PATH", "")
    log("starting backend: " + alias)
    with open("/tmp/llama-backend-stdout.log", "w") as out:
        _proc = subprocess.Popen(cmd, stdout=out, stderr=subprocess.STDOUT, env=env)
    deadline = time.time() + 900
    while time.time() < deadline:
        if healthy():
            _current = alias
            log("backend ready: " + alias)
            return True
        if _proc.poll() is not None:
            log("backend exited early; check /tmp/llama-backend-stdout.log")
            return False
        time.sleep(3)
    log("backend load timeout")
    return False

def ensure_model(alias):
    global _current
    with _lock:
        if alias not in MODEL_CONFIGS:
            return False
        if _current == alias and healthy():
            return True
        stop_backend()
        return start_backend(alias)

def official_webui_available():
    """True when llama-server serves its built-in WebUI at /."""
    try:
        r = requests.get("http://localhost:%d/" % BACKEND_PORT, timeout=8)
        t = r.text
        return r.status_code == 200 and (("<!DOCTYPE html" in t) or ("/assets/" in t) or ("llama" in t.lower()))
    except Exception:
        return False

class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, fmt, *args):
        pass

    def _send_json(self, obj, status=200):
        data = json.dumps(obj, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def _send_html(self, html, status=200):
        data = html.encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_OPTIONS(self):
        self.send_response(204)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "*")
        self.send_header("Content-Length", "0")
        self.end_headers()

    def do_GET(self):
        p = self.path
        if p == "/health":
            self._send_json({"status": "ok", "current_model": _current,
                             "available": list(MODEL_CONFIGS.keys())})
            return
        if p.startswith("/v1/models"):
            data = [{"id": a, "object": "model", "owned_by": "kaggle", "created": 0,
                     "status": {"value": "loaded", "args": []}}
                    for a in MODEL_CONFIGS]
            self._send_json({"object": "list", "data": data})
            return
        if p == "/models/sse":
            self.send_response(200)
            self.send_header("Content-Type", "text/event-stream")
            self.send_header("Cache-Control", "no-cache")
            self.end_headers()
            try:
                for _a in MODEL_CONFIGS:
                    _ev = {"model": _a, "event": "model_status",
                           "data": {"value": "loaded", "args": []}}
                    self.wfile.write(("data: " + json.dumps(_ev) + "\n\n").encode())
                self.wfile.flush()
                for _ in range(600):
                    time.sleep(1)
                    self.wfile.write(b": ping\n\n")
                    self.wfile.flush()
            except Exception:
                pass
            return
        if p == "/v1/model":
            self._send_json({"current_model": _current})
            return
        # Everything else: official llama.cpp WebUI (/, /assets/*, /props, ...)
        self._serve_webui("GET")

    def do_POST(self):
        p = self.path
        if p == "/models/load":
            self._router_load()
            return
        if p == "/models/unload":
            self._send_json({"success": True})
            return
        if p.startswith("/v1/") or p.startswith("/completion"):
            self._forward_api("POST")
        else:
            self._serve_webui("POST")

    def _router_load(self):
        length = int(self.headers.get("Content-Length", 0) or 0)
        raw = self.rfile.read(length) if length else b""
        wanted = None
        if raw:
            try:
                wanted = json.loads(raw).get("model")
            except Exception:
                pass
        if wanted not in MODEL_CONFIGS:
            self._send_json({"error": {"message": "unknown model: " + str(wanted),
                                       "type": "invalid_request_error"}}, 404)
            return
        if ensure_model(wanted):
            self._send_json({"success": True, "model": wanted})
        else:
            self._send_json({"error": {"message": "failed to load " + wanted,
                                       "type": "server_error"}}, 500)

    def _serve_webui(self, method):
        alias = _current if _current in MODEL_CONFIGS else DEFAULT_MODEL
        if not ensure_model(alias):
            self._send_json({"error": {"message": "Backend failed to load " + alias,
                                       "type": "server_error"}}, 503)
            return
        if method == "GET" and self.path in ("/", "/index.html"):
            if official_webui_available():
                log("serving OFFICIAL llama.cpp WebUI")
                self._raw_proxy("GET")
            else:
                log("official WebUI not found; serving fallback chat page")
                self._send_html(FALLBACK_UI)
            return
        self._raw_proxy(method)

    def _raw_proxy(self, method):
        length = int(self.headers.get("Content-Length", 0) or 0)
        body = self.rfile.read(length) if length else b""
        url = "http://localhost:%d%s" % (BACKEND_PORT, self.path)
        fwd = {k: v for k, v in self.headers.items()
               if k.lower() not in ("host", "content-length", "transfer-encoding", "connection")}
        try:
            r = requests.request(method, url, data=body if body else None,
                                 headers=fwd, stream=True, timeout=(10, 3600))
        except Exception as e:
            self._send_json({"error": {"message": str(e), "type": "server_error"}}, 502)
            return
        is_stream = "text/event-stream" in r.headers.get("content-type", "")
        self.send_response(r.status_code)
        for k, v in r.headers.items():
            if k.lower() in ("content-length", "transfer-encoding", "connection", "content-encoding"):
                continue
            self.send_header(k, v)
        if is_stream:
            self.send_header("Transfer-Encoding", "chunked")
            self.end_headers()
            try:
                for chunk in r.iter_content(chunk_size=1024):
                    if chunk:
                        self.wfile.write(("%X\r\n" % len(chunk)).encode() + chunk + b"\r\n")
                        self.wfile.flush()
                self.wfile.write(b"0\r\n\r\n")
                self.wfile.flush()
            except Exception:
                pass
        else:
            payload = r.content
            self.send_header("Content-Length", str(len(payload)))
            self.end_headers()
            self.wfile.write(payload)

    def _forward_api(self, method):
        length = int(self.headers.get("Content-Length", 0) or 0)
        body = self.rfile.read(length) if length else b""
        wanted = None
        if body:
            try:
                wanted = json.loads(body).get("model")
            except Exception:
                pass
        alias = wanted if wanted in MODEL_CONFIGS else (DEFAULT_MODEL if wanted is None else None)
        if alias is None:
            self._send_json({"error": {"message": "Unknown model: " + str(wanted),
                                       "type": "invalid_request_error"}}, 400)
            return
        if not ensure_model(alias):
            self._send_json({"error": {"message": "Failed to load model " + alias,
                                       "type": "server_error"}}, 500)
            return
        url = "http://localhost:%d%s" % (BACKEND_PORT, self.path)
        fwd = {k: v for k, v in self.headers.items()
               if k.lower() not in ("host", "content-length", "transfer-encoding", "connection")}
        try:
            r = requests.request(method, url, data=body if body else None,
                                 headers=fwd, stream=True, timeout=(10, 3600))
        except Exception as e:
            self._send_json({"error": {"message": str(e), "type": "server_error"}}, 502)
            return
        is_stream = "text/event-stream" in r.headers.get("content-type", "")
        self.send_response(r.status_code)
        for k, v in r.headers.items():
            if k.lower() in ("content-length", "transfer-encoding", "connection", "content-encoding"):
                continue
            self.send_header(k, v)
        if is_stream:
            self.send_header("Transfer-Encoding", "chunked")
            self.end_headers()
            try:
                for chunk in r.iter_content(chunk_size=1024):
                    if chunk:
                        self.wfile.write(("%X\r\n" % len(chunk)).encode() + chunk + b"\r\n")
                        self.wfile.flush()
                self.wfile.write(b"0\r\n\r\n")
                self.wfile.flush()
            except Exception:
                pass
        else:
            payload = r.content
            self.send_header("Content-Length", str(len(payload)))
            self.end_headers()
            self.wfile.write(payload)

if __name__ == "__main__":
    log("available models: " + ", ".join(MODEL_CONFIGS.keys()))
    log("default model: " + DEFAULT_MODEL)
    srv = ThreadingHTTPServer(("0.0.0.0", GATEWAY_PORT), Handler)
    log("gateway listening on port " + str(GATEWAY_PORT))
    threading.Thread(target=ensure_model, args=(DEFAULT_MODEL,), daemon=True).start()
    srv.serve_forever()
'''

GATEWAY_PATH = WORKING / 'model_gateway.py'
GATEWAY_PATH.write_text(GATEWAY_SRC)

# Pass configuration to the gateway via environment variables
os.environ['MODEL_CONFIGS'] = json.dumps(MODEL_CONFIGS, ensure_ascii=False)
os.environ['DEFAULT_MODEL'] = DEFAULT_MODEL
os.environ['GATEWAY_PORT'] = str(PORT)
os.environ['BACKEND_PORT'] = str(BACKEND_PORT)
os.environ['LLAMA_SERVER'] = str(LLAMA_SERVER)

# Kill any stale processes, then launch the gateway
subprocess.run(['pkill', '-f', 'model_gateway.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'llama-server'], capture_output=True)
import time as _time
_time.sleep(2)
with open('/tmp/gateway.log', 'w') as _glog:
    gateway_proc = subprocess.Popen(
        ['python', str(GATEWAY_PATH)],
        stdout=_glog, stderr=subprocess.STDOUT, env={**os.environ}
    )
_time.sleep(4)

# Health check
import requests as _req
for _ in range(10):
    try:
        _r = _req.get(f'http://localhost:{PORT}/health', timeout=2)
        if _r.status_code == 200:
            print('✅ Gateway is up:', _r.json())
            break
    except Exception:
        pass
    _time.sleep(1)
else:
    print('❌ Gateway failed. Log:')
    print(open('/tmp/gateway.log').read()[-1500:])
    raise RuntimeError('Gateway did not start')

print(f'   OpenAI endpoint : http://localhost:{PORT}/v1')
print(f'   Models          : {list(MODEL_CONFIGS.keys())}')
print('   Switching       : set the "model" field in your request')

## 🌐 Cell 6 — Start Cloudflare Tunnel & get public URL

In [ ]:
import re, subprocess, time

# Clear any stale tunnel from a previous run so only this run's URL is live.
subprocess.run(["pkill", "-9", "-f", "cloudflared"], capture_output=True)
time.sleep(2)

tunnel_log = "/tmp/cloudflare.log"
tunnel_url = None

with open(tunnel_log, "w") as log:
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
        stdout=log, stderr=log,
    )

def _extract_tunnel_url(text):
    """Extract the public URL from cloudflared log (no regex escaping pitfalls)."""
    for token in text.replace("|", " ").split():
        if "https://" in token and "trycloudflare.com" in token:
            i = token.find("https://")
            j = token.find("trycloudflare.com", i)
            if j > i:
                return token[i:j + len("trycloudflare.com")]
    return None

print("⏳ Waiting for Cloudflare tunnel URL (up to 120s)...")
for _ in range(60):
    time.sleep(2)
    if tunnel_proc.poll() is not None:
        break
    try:
        with open(tunnel_log, errors="ignore") as f:
            tunnel_url = _extract_tunnel_url(f.read())
    except Exception:
        tunnel_url = None
    if tunnel_url:
        break

if not tunnel_url:
    print("❌ Could not get Cloudflare URL. Full log:")
    with open(tunnel_log, errors="ignore") as f:
        print(f.read()[-4000:])
    raise RuntimeError("Cloudflare tunnel failed to start.")

print()
print("═" * 62)
print("  🔗 YOUR SERVER URL（每次重建都会变，务必用这一次打印的新地址！）")
print(f"  {tunnel_url}")
print("═" * 62)
print()
print(f"  OpenAI endpoint : {tunnel_url}/v1")
print(f"  Available models:")
for a in MODEL_CONFIGS:
    print(f"     • {a}")
print()
print(f"  Switch model = change the \"model\" field in your request.")
print(f"  e.g. model=\"{ALIAS_DEEPSEEK}\" for the 9B uncensored model")

## 🔁 Cell 7 — Keep-alive loop

Keeps the session alive (~12h) and prints periodic health checks, including which model is currently loaded.

In [ ]:
import datetime, requests

start_time = datetime.datetime.now()
check_interval = 300
print(f"🟢 Server running since {start_time.strftime('%H:%M:%S')}")
print(f"   URL     : {tunnel_url}")
print(f"   Models  : {list(MODEL_CONFIGS.keys())}")
print(f"   Timeout : ~12h (Kaggle session limit)")
print("   Press ■ (interrupt kernel) to stop.\n" + "─" * 60)

try:
    while True:
        time.sleep(check_interval)
        elapsed = datetime.datetime.now() - start_time
        h, rem = divmod(int(elapsed.total_seconds()), 3600)
        m = rem // 60
        try:
            r = requests.get(f"http://localhost:{PORT}/health", timeout=5)
            info = r.json()
            status = f"✅ healthy | model: {info.get('current_model')}"
        except Exception:
            status = "❌ unreachable"
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"  [{ts}]  Uptime: {h}h {m:02d}m  |  {status}")
        if "❌" in status:
            print("\n⚠️  Gateway unreachable. Check /tmp/gateway.log")
            break
except KeyboardInterrupt:
    print("\n🛑 Shutting down...")
    gateway_proc.terminate()
    tunnel_proc.terminate()
    subprocess.run(["pkill", "-f", "llama-server"], capture_output=True)
    print("   Done.")